## Prepare data for the Stoke ward of Plymouth
- Get anchor loads for Stoke
- Get green space for Stoke
- Get DESNZ HN pilot zones

Resulting datasets will be formatted for plotting in Flourish.

In [1]:
import pandas as pd
import polars as pl
import geopandas as gpd
from asf_heat_pump_suitability.getters import get_datasets
from asf_heat_pump_suitability.pipeline.prepare_features import anchor_properties

In [31]:
from asf_heat_pump_suitability import PROJECT_DIR
import os

output_directory = os.path.join(PROJECT_DIR, "outputs/area_specific_analysis/stoke/")
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

## Load datasets and filter to just Stoke

In [2]:
stoke_ward_boundary = get_datasets.load_stoke_bound().to_crs(epsg=4326)

In [3]:
anchor_properties_df = anchor_properties.load_gdf_and_process_poi().to_crs(epsg=4326)

2025-07-23 14:14:57,974 - asf_heat_pump_suitability.pipeline.prepare_features.anchor_properties - INFO - Loading POI data...
2025-07-23 14:20:38,952 - asf_heat_pump_suitability.pipeline.prepare_features.anchor_properties - INFO - Found 38972 potential anchor properties
2025-07-23 14:20:38,958 - asf_heat_pump_suitability.pipeline.prepare_features.anchor_properties - INFO - Output CRS: EPSG:27700


In [25]:
stoke_anchor_properties_df = gpd.sjoin(
    anchor_properties_df,
    stoke_ward_boundary,
    how="inner",
    predicate="intersects"
)

In [18]:
greenspace_df = get_datasets.load_SX_Greenspace().to_crs(epsg=4326)

In [23]:
stoke_greenspace_df = gpd.sjoin(
    greenspace_df,
    stoke_ward_boundary,
    how="inner",
    predicate="intersects",
)

In [43]:
hnz_plymouth = gpd.read_file("s3://asf-heat-pump-suitability/heat_network_desnz_data/heat-network-zone-map-Plymouth.gpkg").to_crs(epsg=4326)

## Save out for Flourish plotting
- Green space: regions
- Anchor loads: points, need to be in same format as clustered points currently are.

In [35]:
stoke_greenspace_df_formatted = stoke_greenspace_df[['function', 'distName1', 'geometry']].rename(
    columns={'function': 'Greenspace type', 'distName1': 'Greenspace name'})
stoke_greenspace_df_formatted["Type"] = "Greenspace"

stoke_greenspace_df_formatted.to_file(
    os.path.join(output_directory, "stoke_greenspace.geojson"),
    driver="GeoJSON"
)

2025-07-23 14:53:53,555 - pyogrio._io - INFO - Created 17 records


In [42]:
stoke_anchor_properties_df_formatted = stoke_anchor_properties_df[['main_category', 'geometry']].rename(
    columns={'main_category': 'Anchor load type'})
stoke_anchor_properties_df_formatted["Type"] = "Anchor load" # this is the name + colour in flourish
stoke_anchor_properties_df_formatted["Scale"] = 2 # For different scales?

# The clustered points are currently given a bit of jitter
stoke_anchor_properties_df_formatted['Long'] = stoke_anchor_properties_df_formatted['geometry'].x
stoke_anchor_properties_df_formatted['Lat'] = stoke_anchor_properties_df_formatted['geometry'].y

stoke_anchor_properties_df_formatted.to_file(
    os.path.join(output_directory, "stoke_anchorloads.geojson"),
    driver="GeoJSON"
)

2025-07-23 15:13:46,543 - pyogrio._io - INFO - Created 10 records


In [46]:
# Don't filter plymouth HN zones for Stoke - might be nice to see the ones that are close, but not in Stoke
hnz_plymouth["Type"] = "DESNZ HN zone"

hnz_plymouth[['geometry', 'Type']].to_file(
    os.path.join(output_directory, "stoke_desnz_hn.geojson"),
    driver="GeoJSON"
)

2025-07-24 15:57:49,959 - pyogrio._io - INFO - Created 12 records
